# ДЗ №1. Задание 1
### Выполнил: Лёвин В.А.
### Группа: ИУ6-54Б

Расстояния между заведениями.


In [105]:
import csv
from math import asin, cos, radians, sin, sqrt
from pyspark import SparkConf, SparkContext

conf = SparkConf().setMaster("local[2]").setAppName("HW1-Places")
sc = SparkContext.getOrCreate(conf=conf)

## Чтение данных

```python
# Наименование столбцов
['ID', 'Name', 'global_id', 'IsNetObject', 'OperatingCompany', 'TypeObject', 'AdmArea', 'District', 'Address', 'PublicPhone', 'SeatsCount', 'SocialPrivileges', 'Longitude_WGS84', 'Latitude_WGS84', 'geoData']
```

`mapPartitions` разбирает строки каждой части RDD. `cache` сохраняет прочитанные записи для повторных вычислений.


In [2]:
def parse_place(row):
    return {
        "id": int(row[0]),
        "name": row[1],
        "address": row[8],
        "lat": float(row[13]),
        "lon": float(row[12]),
    }

places = (
    sc.textFile("places.csv")
    .mapPartitions(csv.reader)
    .map(parse_place)
    .cache()
)
print(f"Заведений: {places.count()}")

Заведений: 769


## Формула расстояния

$$a = \sin^2\frac{\varphi_2-\varphi_1}{2} + \cos\varphi_1\cos\varphi_2\sin^2\frac{\lambda_2-\lambda_1}{2},$$
$$d = 2R\arcsin\sqrt{a}.$$




In [3]:
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(radians, (lat1, lon1, lat2, lon2))
    a = (
        sin((lat2 - lat1) / 2) ** 2
        + cos(lat1) * cos(lat2) * sin((lon2 - lon1) / 2) ** 2
    )
    return 2 * 6_371_000 * asin(sqrt(min(1.0, a)))

## 1. Расстояния от заданной точки

Для каждого заведения считаем расстояние от точки `(55.751244, 37.618423)`. `take(10)` выводит первые записи RDD без сортировки.


In [4]:
def show_places(rows):
    print(f"{'ID':<8} {'Название':<40} {'Расстояние, м':>14} \tАдрес")
    for place, distance in rows:
        print(f"{place['id']:<8} {place['name']:<40} {distance:>14.2f} \t{place['address']}")

point_lat, point_lon = 55.751244, 37.618423
def distance_from_point(place):
    distance = haversine(point_lat, point_lon, place["lat"], place["lon"])
    return place, distance

point_distances = places.map(distance_from_point).cache()
show_places(point_distances.take(10))


ID       Название                                  Расстояние, м 	Адрес
21830    Шоколадница                                     4481.04 	город Москва, Бауманская улица, дом 33/2, строение 1
22366    МУ-МУ                                           4411.45 	город Москва, Бауманская улица, дом 35/1
28608    КОМБИНАТ ПИТАНИЯ МГТУ ИМ.Н.Э.БАУМАНА            4141.57 	город Москва, Бригадирский переулок, дом 3-5
132486   Дом 12                                          2976.78 	город Москва, улица Казакова, дом 8А
22357    Чито-Ра                                         3034.84 	город Москва, улица Казакова, дом 10/2, строение 1
21659    Бар- буфет «Николай»                            2828.73 	город Москва, Старая Басманная улица, дом 5, строение 1
26624    Флорентини                                      2888.58 	город Москва, Старая Басманная улица, дом 9, корпус 1
156563   Beer Gik                                        1565.31 	город Москва, Мясницкая улица, дом 22, строение 1
22129    Погр

## 2. Расстояния между заведениями
Условие `id1 < id2` исключает расстояние до самого себя и повтор пары. 


In [103]:
def is_unique_pair(pair):
    first, second = pair
    return first["id"] < second["id"]

def distance_between_places(pair):
    first, second = pair
    distance = haversine(first["lat"], first["lon"], second["lat"], second["lon"])
    return first, second, distance

def show_pairs(rows):
    print(f"{'Заведение 1':<25} {'Заведение 2':<45} {'Расстояние, м':<14}")
    for first, second, distance in rows:
        name1 = f"{first['id']} {first['name']}"
        name2 = f"{second['id']} {second['name']}"
        print(f"{name1:<25} {name2:<45} {distance:<14.2f}")
        
pairs = places.cartesian(places).filter(is_unique_pair)
pair_distances = pairs.map(distance_between_places).cache()

show_pairs(pair_distances.take(10))

Заведение 1               Заведение 2                                   Расстояние, м 
21830 Шоколадница         22366 МУ-МУ                                   161.32        
21830 Шоколадница         28608 КОМБИНАТ ПИТАНИЯ МГТУ ИМ.Н.Э.БАУМАНА    844.24        
21830 Шоколадница         132486 Дом 12                                 1580.44       
21830 Шоколадница         22357 Чито-Ра                                 1559.11       
21830 Шоколадница         26624 Флорентини                              1594.26       
21830 Шоколадница         156563 Beer Gik                               3025.43       
21830 Шоколадница         22129 Погребок                                3025.43       
21830 Шоколадница         150572 Пробка Гриль                           2936.94       
21830 Шоколадница         154627 TEMPO DI PASTA                         2941.07       
21830 Шоколадница         154750 Хлеб насущный                          2941.07       


26/11/15 15:12:37 WARN BlockManager: Task 179 already completed, not releasing lock for rdd_251_0


## 3. Ближайшие и удалённые пары заведений

`takeOrdered` выбирает наименьшие расстояния, `top` — наибольшие.

In [104]:
def by_distance(row):
    return row[2]

nearest_pairs = pair_distances.takeOrdered(10, key=by_distance)
farthest_pairs = pair_distances.top(10, key=by_distance)

print("10 наиболее близких пар заведений")
show_pairs(nearest_pairs)
print("\n10 наиболее удалённых пар заведений")
show_pairs(farthest_pairs)


10 наиболее близких пар заведений
Заведение 1               Заведение 2                                   Расстояние, м 
22129 Погребок            156563 Beer Gik                               0.00          
154627 TEMPO DI PASTA     154750 Хлеб насущный                          0.00          
156563 Beer Gik           166443 Kozlovna                               0.00          
22129 Погребок            166443 Kozlovna                               0.00          
156563 Beer Gik           172396 Па-Паэлья                              0.00          
22129 Погребок            172396 Па-Паэлья                              0.00          
120714 Глав Пив Маг       146555 Beermood                               0.00          
143164 Блэк милк          151758 Дабл Би                                0.00          
157308 Jeffreys coffee    170123 Мегобари                               0.00          
143164 Блэк милк          167828 Коленки Пчелы                          0.00          

10 наибо